# Case
**Convert vector data with continuous values into raster format**

The labels are either based on a provided text- or Esri-shapefile.<br>
To spatially align the labels with other data sets, a template raster (base raster) can be provided.


In [21]:
import rasterio
import geopandas as gpd
import numpy as np

from tqdm import tqdm
from beak.experimental.conversions import create_continuous_raster
from beak.experimental.io import data_folder, check_path

# Set base data folder path
data_folder = data_folder()
vector_file_name = "GBWDB_Rock"

# Paths
cma_name = "regional_tusk_great_basin_a2_102008_100"
base_raster_file = data_folder / "processed" / cma_name / "base_raster" / "template_raster_geodawn_area2.tif"
vector_file = data_folder / "raw" / "geology" / "Lederer_2020_Tungsten_Skarn_Great_Basin" / "Shapefiles" / str(vector_file_name + ".shp")

out_folder = data_folder / "processed" / cma_name / "labels"
check_path(out_folder)


WindowsPath('S:/Projekte/20230082_DARPA_CriticalMAAS_TA3/Bearbeitung/GitHub/beak-ta3/src/beak/data/processed/regional_tusk_great_basin_a2_102008_100/labels')

# Create Continuous Raster

In [6]:
# Load shapefile and create geodataframe
base_raster = rasterio.open(base_raster_file)

gdf = gpd.read_file(vector_file)
gdf = gdf.to_crs(base_raster.crs)

In [27]:
# Create raster without snap. Remove the `snap_to_origin=None` statement or provide a base raster for automatic alignment.
columns = ["W_ppm"]
nodata = -99999

for column in tqdm(columns):
    QUERY = f"{column} > 0"
    raster_array, raster_meta = create_continuous_raster(
        geodataframe=gdf,
        column=column,
        base_raster=base_raster,
        query=QUERY,
        out_file=str(out_folder / (vector_file_name + "_" + column + ".tif").lower()),
        nodata=nodata,
    )
    
    vector_points = len(gdf.query(QUERY))
    rasterized_points = np.sum(raster_array > raster_meta["nodata"])
    print(f"Column {column} has {rasterized_points} rasterized labels (from {vector_points}).")



100%|██████████| 1/1 [00:00<00:00,  5.68it/s]

File already exists: gbwdb_rock_w_ppm.tif.
Column W_ppm has 802 rasterized labels (from 5808).
